In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="01-ml-design-prep",
    notebook_name="01_ml_design_framework.ipynb"
)

# ML System Design Interview -- The Complete 7-Step Framework

## The Big Idea (For a 12-Year-Old)

Imagine your school principal asks: *"Build a library system that recommends books every student will love."*

You wouldn't just say "use a neural network." You'd need to think through the WHOLE thing:
- What does "a good recommendation" even mean? *(business objective)*
- What data do we have about students and books? *(data preparation)*
- How do we turn "book-liking" into math a computer can solve? *(ML framing)*
- How do we build and test it? *(model development + evaluation)*
- How do we get it running for every student in real time? *(deployment)*
- What happens when reading trends change or a new grade joins? *(monitoring)*

**That is exactly what an ML system design interview asks you to do** -- for real products like YouTube, Instagram, Spotify, or Google Maps.

---

## Staff-Level Technical Summary

This notebook covers the **7-step ML system design framework** used across all case studies in this repo:

1. **Clarify Requirements** -- structured question taxonomy with examples
2. **Frame as ML Task** -- business-to-ML objective translation, input/output spec, ML categories
3. **Data Preparation** -- data engineering + feature engineering with runnable code
4. **Model Development** -- baseline-to-complex progression, training strategies
5. **Evaluation** -- offline + online metric reference tables with implementations
6. **Deployment & Serving** -- cloud vs. on-device, A/B testing, batch vs. online prediction
7. **Monitoring** -- distribution shift, retraining triggers, what to watch

Each section has interactive Python examples you can run.

## The 7-Step Framework at a Glance

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('The 7-Step ML System Design Framework', fontsize=18, fontweight='bold', pad=20)

steps = [
    (1, "1. Clarify\nRequirements",  "#FFE0B2", "Ask: business objective, scale,\ndata, constraints, latency"),
    (2, "2. Frame as\nML Task",       "#B3E5FC", "Define: ML objective, input/output,\nML category (ranking/classification/regression)"),
    (3, "3. Data\nPreparation",       "#C8E6C9", "Data engineering + feature engineering\n(sources, ETL, encoding, scaling)"),
    (4, "4. Model\nDevelopment",      "#F8BBD0", "Baseline → simple → complex\nLoss functions, training strategies"),
    (5, "5. Evaluation",               "#D1C4E9", "Offline (precision, AUC, nDCG)\nOnline (CTR, revenue, engagement)"),
    (6, "6. Deployment\n& Serving",   "#FFF9C4", "Cloud vs on-device, batch vs online,\nA/B testing, model compression"),
    (7, "7. Monitoring\n& Infra",     "#FFCCBC", "Distribution shift, retraining,\noperational metrics"),
]

for (i, label, color, desc) in steps:
    y = 7.5 - (i - 1) * 1.0
    box = mpatches.FancyBboxPatch((0.3, y - 0.35), 2.2, 0.7,
        boxstyle='round,pad=0.05', facecolor=color, edgecolor='#333', linewidth=2)
    ax.add_patch(box)
    ax.text(1.4, y, label, ha='center', va='center', fontsize=10, fontweight='bold')
    ax.text(3.0, y, desc, ha='left', va='center', fontsize=9, color='#444')
    if i < 7:
        ax.annotate('', xy=(1.4, y - 0.36), xytext=(1.4, y - 0.35 + 0.0),
                    arrowprops=dict(arrowstyle='->', color='#666', lw=1.5))

# Time allocation bar
times = [5, 5, 10, 12, 5, 5, 3]
labels = ['Clarify', 'Frame', 'Data', 'Model', 'Eval', 'Deploy', 'Monitor']
colors = [s[2] for s in steps]
ax2 = fig.add_axes([0.55, 0.1, 0.4, 0.8])
bars = ax2.barh(labels[::-1], times[::-1], color=colors[::-1], edgecolor='#333', linewidth=1.5)
ax2.set_xlabel('Minutes in 45-minute interview', fontsize=10)
ax2.set_title('Recommended Time Allocation', fontsize=12, fontweight='bold')
for bar, t in zip(bars, times[::-1]):
    ax2.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
             f'{t} min', va='center', fontsize=9)
ax2.set_xlim(0, 18)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('framework_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print("Total: 45 minutes")

## Step 1: Clarifying Requirements

### The 12-Year-Old Version

Imagine someone says: *"Build me a cool robot."* Before you start, you'd ask:
- What should it do? How big? Does it fly? How much can we spend?

ML design questions are **intentionally vague**. "Design a recommendation system" could mean TikTok's feed, Spotify playlists, or Amazon product suggestions. Always clarify first.

### The 6 Question Categories

In [ ]:
import pandas as pd

categories = pd.DataFrame([
    ["Business Objective",    "What is the ultimate goal?",             "Increase bookings? Revenue? Retention?"],
    ["System Features",       "What interactions/actions exist?",       "Can users like/dislike? Rate? Share?"],
    ["Data",                  "What data sources exist? How much?",     "User-generated? System logs? Labels?"],
    ["Constraints",           "Compute budget? Cloud or on-device?",    "Latency? Privacy? Model size limit?"],
    ["Scale",                 "Users? Items? Growth rate?",             "10M users? 1B videos? 1K QPS?"],
    ["Performance",           "Latency? Accuracy vs speed?",            "Real-time (<200ms)? Batch OK?"],
], columns=["Category", "What to Ask", "Example Probes"])

print("=== Clarifying Requirements Question Taxonomy ===")
print(categories.to_string(index=False))
print()
print("Pro tip: After clarifying, WRITE DOWN the requirements.")
print("Say: 'Let me summarize what I heard...' -- this signals thoroughness.")

## Step 2: Frame the Problem as an ML Task

### Business Objective → ML Objective Translation

A business goal like "increase revenue" isn't directly trainable. You need to translate it.

In [ ]:
translations = pd.DataFrame([
    ["Video streaming",    "Increase watch time",       "Predict P(user watches ≥50% of video)",   "Ranking"],
    ["Social platform",   "Improve safety",            "Predict P(post is harmful)",               "Binary classification"],
    ["E-commerce",        "Increase sales",            "Predict P(user purchases item)",           "Ranking / Binary clf"],
    ["Ad platform",       "Increase ad revenue",       "Predict P(user clicks ad)",                "Binary classification"],
    ["Maps",              "Accurate ETAs",             "Predict travel time (minutes)",            "Regression"],
    ["Social network",    "Grow connections",          "Predict P(user accepts friend request)",   "Binary classification"],
], columns=["Application", "Business Objective", "ML Objective", "ML Category"])

print("=== Business → ML Objective Translation ===")
print(translations.to_string(index=False))

In [ ]:
# ML Category Taxonomy
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 14)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_title('ML Category Taxonomy', fontsize=16, fontweight='bold')

def box(ax, x, y, w, h, text, color, fontsize=9):
    p = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
        facecolor=color, edgecolor='#333', linewidth=1.5)
    ax.add_patch(p)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=fontsize, fontweight='bold', wrap=True)

box(ax, 5.5, 4.5, 3, 1.0, 'Machine Learning', '#E3F2FD', 11)

# Level 1
for x, label, color in [(0.2, 'Supervised
Learning', '#C8E6C9'),
                          (4.8, 'Unsupervised
Learning', '#FFE0B2'),
                          (9.5, 'Reinforcement
Learning', '#F8BBD0')]:
    box(ax, x, 3.0, 3.5, 1.0, label, color, 10)

# Level 2 - supervised
for x, label, color in [(0.0, 'Classification', '#A5D6A7'),
                          (1.8, 'Regression', '#A5D6A7'),
                          (3.6, 'Ranking', '#A5D6A7')]:
    box(ax, x, 1.2, 1.6, 1.0, label, color, 9)
# Level 2 - unsupervised
for x, label, color in [(4.8, 'Clustering', '#FFCC80'),
                          (6.6, 'Dim. Reduction', '#FFCC80')]:
    box(ax, x, 1.2, 1.6, 1.0, label, color, 9)

# Real-world examples
examples = [
    (0.0, 0.0, 'Spam detection
Harm detection
Fraud detection'),
    (1.8, 0.0, 'ETA prediction
House prices
Dwell time'),
    (3.6, 0.0, 'Search ranking
Feed ranking
Recommendations'),
]
for x, y, text in examples:
    ax.text(x + 0.8, y + 0.5, text, ha='center', va='center', fontsize=7, color='#555', style='italic')

plt.tight_layout()
plt.show()

## Step 3: Data Preparation

### Data Engineering (Getting Data Ready)

Think of this like a kitchen: before cooking, you need to buy groceries (collect), wash and chop them (transform), and put them in the right containers (store).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.impute import SimpleImputer

# =====================================================================
# Simulate a messy real-world dataset (users with mixed data types)
# =====================================================================
np.random.seed(42)
n = 200

data = pd.DataFrame({
    'user_id':     range(n),
    'age':         np.random.randint(13, 70, n).astype(float),
    'income':      np.random.lognormal(10, 1, n),   # skewed
    'country':     np.random.choice(['US', 'UK', 'IN', 'BR', 'CN'], n),
    'device':      np.random.choice(['mobile', 'desktop', 'tablet', None], n),
    'clicks_7d':   np.random.poisson(15, n).astype(float),
})

# Inject missing values (real-world datasets are never clean)
missing_idx = np.random.choice(n, 30, replace=False)
data.loc[missing_idx[:15], 'age'] = np.nan
data.loc[missing_idx[15:], 'income'] = np.nan

print("=== Raw Data Sample ===")
print(data.head(8).to_string(index=False))
print(f"\nMissing values:")
print(data.isnull().sum())

In [ ]:
# =====================================================================
# Feature Engineering Pipeline
# =====================================================================

df = data.copy()

# 1. Handle missing values
df['age'] = df['age'].fillna(df['age'].median())
df['income'] = df['income'].fillna(df['income'].median())
df['device'] = df['device'].fillna('unknown')

# 2. Feature scaling
scaler_std = StandardScaler()
scaler_mm  = MinMaxScaler()
df['age_scaled']    = scaler_std.fit_transform(df[['age']])
df['clicks_scaled'] = scaler_mm.fit_transform(df[['clicks_7d']])

# 3. Log transformation for skewed features
df['log_income'] = np.log1p(df['income'])

# 4. Bucketization (discretization)
df['age_bucket'] = pd.cut(df['age'],
    bins=[0, 18, 25, 35, 50, 100],
    labels=['<18', '18-25', '25-35', '35-50', '50+'])

# 5. One-hot encoding for low-cardinality categoricals
device_ohe = pd.get_dummies(df['device'], prefix='device')
df = pd.concat([df, device_ohe], axis=1)

# 6. Embedding for high-cardinality categoricals (concept)
# In PyTorch: nn.Embedding(num_countries, embed_dim)
# We just show the idea here with label encoding
le = LabelEncoder()
df['country_id'] = le.fit_transform(df['country'])

print("=== After Feature Engineering ===")
show_cols = ['age', 'age_scaled', 'age_bucket', 'log_income', 'clicks_scaled',
             'country', 'country_id', 'device_mobile', 'device_desktop']
print(df[show_cols].head(6).to_string(index=False))
print()
print("Feature Engineering Checklist:")
print("  [x] Handle missing values (imputation)")
print("  [x] Scale numerical features (standardization / min-max)")
print("  [x] Log-transform skewed distributions")
print("  [x] Bucketize continuous features")
print("  [x] One-hot encode low-cardinality categoricals")
print("  [ ] Embedding layers for high-cardinality categoricals (done in model)")

## Step 4: Model Development

### The Progression: Simple → Complex

Think of it like cooking. Start with a sandwich (quick, easy to check if edible), then try a full meal, then a gourmet dinner. Don't start with the gourmet dinner -- you'll waste a lot of ingredients.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')

# Simulate a binary classification task (e.g., "will user click?")
np.random.seed(42)
n = 2000
X = np.random.randn(n, 10)
# True signal: clicks depend on features 0, 2, 5 with non-linear interactions
logits = 0.5*X[:,0] + 0.3*X[:,2] - 0.4*X[:,5] + 0.2*X[:,0]*X[:,2]
y = (1 / (1 + np.exp(-logits)) > 0.5).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

results = {}

# ---- Baseline: Logistic Regression ----
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
results['Logistic Regression\n(Baseline)'] = roc_auc_score(y_test, lr.predict_proba(X_test)[:,1])

# ---- Simple: Random Forest ----
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
results['Random Forest\n(Simple)'] = roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])

# ---- Complex: GBDT ----
gbdt = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
gbdt.fit(X_train, y_train)
results['GBDT / XGBoost\n(Complex)'] = roc_auc_score(y_test, gbdt.predict_proba(X_test)[:,1])

# ---- Deep: Simple Neural Network ----
class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(10, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.net(x).squeeze(-1)

model = SimpleNN()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
Xt = torch.FloatTensor(X_train); yt = torch.FloatTensor(y_train)
for _ in range(100):
    pred = model(Xt); loss = nn.BCELoss()(pred, yt)
    opt.zero_grad(); loss.backward(); opt.step()
model.eval()
with torch.no_grad():
    nn_preds = model(torch.FloatTensor(X_test)).numpy()
results['Neural Network\n(Deep)'] = roc_auc_score(y_test, nn_preds)

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 5))
names = list(results.keys())
aucs  = list(results.values())
colors = ['#FFB74D', '#81C784', '#64B5F6', '#BA68C8']
bars = ax.bar(names, aucs, color=colors, edgecolor='#333', linewidth=1.5, width=0.5)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel('ROC-AUC', fontsize=12)
ax.set_title('Model Progression: Baseline → Complex', fontsize=14, fontweight='bold')
for bar, auc in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{auc:.3f}', ha='center', fontsize=11, fontweight='bold')
ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Random baseline')
ax.legend(); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()
print("Key lesson: Start simple. Only go complex when simple doesn't work.")

## Step 5: Evaluation

### Offline vs. Online Metrics

**Offline metrics** are measured during development before deployment (on a test set).
**Online metrics** are measured in production with real users (via A/B tests).

The gap between offline and online is a common interview trap\!

In [ ]:
# Reference table of all common metrics by task type
metrics_ref = pd.DataFrame([
    # Classification
    ["Classification", "Precision",       "TP / (TP+FP)",       "Of positives predicted, how many are correct?"],
    ["Classification", "Recall",          "TP / (TP+FN)",       "Of actual positives, how many did we find?"],
    ["Classification", "F1 Score",        "2*P*R / (P+R)",      "Harmonic mean of precision and recall"],
    ["Classification", "ROC-AUC",         "Area under ROC curve","Overall classification performance (1=perfect)"],
    ["Classification", "PR-AUC",          "Area under PR curve", "Better for imbalanced datasets"],
    # Regression
    ["Regression",     "MAE",             "Mean |y - ŷ|",        "Average absolute error"],
    ["Regression",     "RMSE",            "sqrt(Mean (y-ŷ)²)",   "Penalizes large errors more"],
    ["Regression",     "Huber Loss",      "Hybrid MAE+MSE",      "Robust to outliers"],
    # Ranking
    ["Ranking",        "Precision@k",     "Relevant in top-k / k","Fraction of top-k that are relevant"],
    ["Ranking",        "Recall@k",        "Relevant in top-k / total relevant", "Coverage of relevant items"],
    ["Ranking",        "MRR",             "Mean(1/rank of first hit)", "Rank of first relevant result"],
    ["Ranking",        "mAP",             "Mean Average Precision","Ranking quality with binary relevance"],
    ["Ranking",        "nDCG",            "DCG / IDCG",          "Ranking quality with graded relevance"],
    # Object Detection
    ["Detection",      "IoU",             "Overlap / Union",     "Bounding box overlap quality"],
    ["Detection",      "mAP@IoU",         "Mean AP at IoU thresh","Standard object detection metric"],
    # Embeddings
    ["Embeddings",     "Recall@k",        "Correct in top-k / total","Retrieval accuracy"],
    ["Embeddings",     "Cosine Similarity","cos(a,b)",           "Similarity between embedding vectors"],
], columns=["Task Type", "Metric", "Formula", "When to Use"])

print("=== Offline Metrics Reference ===")
print(metrics_ref.to_string(index=False))

In [ ]:
# Online metrics by system type
online_metrics = pd.DataFrame([
    ["Search / Retrieval",   "Click-Through Rate (CTR)",       "Clicks / Impressions"],
    ["Search / Retrieval",   "Query Success Rate",              "Queries with clicks / Total queries"],
    ["Recommendation",       "CTR + Watch Time",                "Clicks + avg watch time on recommended items"],
    ["Recommendation",       "Diversity",                       "Avg pairwise dissimilarity of recommendations"],
    ["Harmful Content",      "Harmful Impressions",             "Harmful posts seen / Total impressions"],
    ["Harmful Content",      "Valid Appeals Rate",               "Reversed removals / Total removals"],
    ["Ad Click",             "Revenue per mille (RPM)",         "Revenue per 1000 impressions"],
    ["Ad Click",             "CTR + Conversion Rate",           "Clicks + downstream purchases"],
    ["News Feed",            "DAU / Time Spent",                "Daily active users, avg session length"],
    ["Friend Recommendation","Requests Sent + Accepted",        "How many friend requests were accepted"],
], columns=["System", "Online Metric", "Formula / Definition"])

print("=== Online Metrics by System Type ===")
print(online_metrics.to_string(index=False))
print()
print("KEY INSIGHT: Offline AUC going up does NOT always mean online metrics improve.")
print("Always A/B test before declaring victory\!")

## Step 6: Deployment & Serving

### Cloud vs. On-Device

Like deciding: should the chef cook at the restaurant (cloud) or give everyone the recipe (on-device)?

In [ ]:
# Comparison table
deployment = pd.DataFrame([
    ["Simplicity",          "Simple -- deploy once",         "Complex -- app updates needed"],
    ["Cost",                "Ongoing cloud costs",           "No cloud costs"],
    ["Network Latency",     "Present (data travels)",        "None"],
    ["Privacy",             "User data goes to server",      "Data stays on device"],
    ["Model Size Limit",    "Virtually unlimited",           "MB-scale limit"],
    ["Offline Support",     "Requires internet",             "Works offline"],
    ["Update Frequency",    "Easy -- update server",         "Requires app store release"],
    ["Hardware",            "Powerful GPUs",                 "Limited CPU/NPU"],
], columns=["Dimension", "Cloud", "On-Device"])

print("=== Cloud vs. On-Device Deployment ===")
print(deployment.to_string(index=False))
print()
print("Rule of thumb:")
print("  - Large models, privacy-insensitive tasks => Cloud")
print("  - Small models, privacy-sensitive, offline-needed => On-Device")
print("  - Example Cloud: YouTube recommendations, Google Search")
print("  - Example On-Device: Siri, Face ID, keyboard autocomplete")

In [ ]:
# Batch vs. Online Prediction
prediction_modes = pd.DataFrame([
    ["Batch Prediction",
     "Pre-compute predictions offline (e.g., nightly)",
     "News feed pre-ranking, email campaigns",
     "No latency at serve time",
     "Can't respond to real-time events; stale predictions"],
    ["Online Prediction",
     "Compute predictions at request time",
     "Search ranking, ad CTR, visual search",
     "Fresh, context-aware predictions",
     "Model must be fast enough for real-time serving"],
], columns=["Mode", "How It Works", "Use Cases", "Pros", "Cons"])

print("=== Batch vs. Online Prediction ===")
print(prediction_modes.to_string(index=False))

## Step 7: Monitoring & Infrastructure

### Why Systems Fail in Production

The #1 reason: **Data distribution shift** -- the data the model sees in production differs from training data.

Example: You trained a CTR model in January. In March, a major news event changes what users click. The model's accuracy drops -- but you don't know unless you're watching.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simulate distribution shift
np.random.seed(42)
days = np.arange(90)

# Training distribution: users mostly on mobile
train_mobile_ratio = 0.6

# Production: mobile usage increases over time (shift\!)
prod_mobile_ratio = 0.6 + 0.003 * days  # gradual drift

# Model performance degrades with drift
base_auc = 0.82
model_auc = base_auc - 0.0015 * days + np.random.normal(0, 0.005, len(days))
model_auc = np.clip(model_auc, 0.6, 0.85)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Feature distribution drift
ax1.plot(days, [train_mobile_ratio]*90, 'b--', linewidth=2, label='Training (mobile ratio)')
ax1.plot(days, prod_mobile_ratio, 'r-', linewidth=2, label='Production (mobile ratio)')
ax1.fill_between(days, train_mobile_ratio, prod_mobile_ratio, alpha=0.2, color='orange')
ax1.set_xlabel('Days in Production')
ax1.set_ylabel('Mobile User Ratio')
ax1.set_title('Feature Distribution Shift', fontsize=13, fontweight='bold')
ax1.legend()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Plot 2: Model performance degradation
ax2.plot(days, model_auc, 'b-', linewidth=2, label='Model AUC')
ax2.axhline(base_auc, color='green', linestyle='--', linewidth=1.5, label=f'Initial AUC = {base_auc}')
ax2.axhline(0.75, color='red', linestyle='--', linewidth=1.5, label='Alert threshold = 0.75')
alert_day = next(i for i, a in enumerate(model_auc) if a < 0.75)
ax2.axvline(alert_day, color='orange', linewidth=2, label=f'Alert triggered: day {alert_day}')
ax2.set_xlabel('Days in Production')
ax2.set_ylabel('ROC-AUC')
ax2.set_title('Model Performance Degradation', fontsize=13, fontweight='bold')
ax2.legend()
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("Monitoring Checklist:")
print("  [x] Track model performance (AUC, precision, recall) over time")
print("  [x] Monitor input feature distributions (detect drift)")
print("  [x] Set alert thresholds -- trigger retraining automatically")
print("  [x] Log operational metrics (latency, throughput, error rates)")
print("  [x] Monitor business metrics (CTR, revenue, engagement)")

## Common Pitfalls to Avoid

These are the 7 most common mistakes in ML design interviews:

In [ ]:
pitfalls = pd.DataFrame([
    ["Jumping to model selection",
     ""I'd use a Transformer for this."",
     "Always clarify requirements first. The interviewer will mark you down."],
    ["Unstructured answers",
     "Rambling across topics without flow",
     "Use the 7-step framework. Tell the interviewer your plan upfront."],
    ["Ignoring business objective",
     "Designing a model that doesn't help the business",
     "Always connect ML objective to business objective explicitly."],
    ["No trade-off discussion",
     ""I'd use BERT" without discussing alternatives",
     "For every decision, mention at least one alternative and explain trade-offs."],
    ["Skipping data discussion",
     "Assuming perfect labeled data exists",
     "Discuss sources, quality, labeling strategy, biases, class imbalance."],
    ["Forgetting production",
     "Designing a model you can't serve at scale",
     "Always discuss latency, serving architecture, and monitoring."],
    ["Only talking about offline metrics",
     ""I'd optimize for AUC" with no online metrics",
     "Always pair offline metrics (AUC, nDCG) with online metrics (CTR, revenue)."],
], columns=["Pitfall", "What It Sounds Like", "How to Fix It"])

print("=== 7 Common Interview Pitfalls ===")
for i, row in pitfalls.iterrows():
    print(f"\n{i+1}. {row['Pitfall']}")
    print(f"   What it sounds like: {row['What It Sounds Like']}")
    print(f"   Fix: {row['How to Fix It']}")

## Interview Cheat Sheet

```
FRAMEWORK (say this at the start):
  "I'll walk through this in 7 steps: requirements, ML framing, data,
   model, evaluation, serving, and monitoring. Let me start by clarifying
   the requirements..."

STEP 1 - CLARIFY (5 min):
  Ask: business objective, scale, latency, data available, interactions

STEP 2 - FRAME (5 min):
  State ML objective, input/output, ML category (classification/ranking/regression)

STEP 3 - DATA (8-10 min):
  Sources → ETL → feature engineering (missing values, scaling, encoding)

STEP 4 - MODEL (10-12 min):
  Baseline → simple → complex. Name loss function. Discuss trade-offs.

STEP 5 - EVAL (5 min):
  Offline metric (AUC/nDCG/mAP) + Online metric (CTR/revenue/engagement)

STEP 6 - SERVING (5 min):
  Cloud vs on-device. Batch vs online. A/B testing. Latency budget.

STEP 7 - MONITORING (3 min):
  Distribution shift. Retraining frequency. Alert thresholds.

KEY PHRASES:
  - "The trade-off here is..."
  - "I'd start with X as a baseline because..."
  - "The key challenge at scale is..."
  - "The offline metric is X but the online metric that matters is Y"
  - "If I had more time I'd also explore..."
```

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)